In [ ]:
print("Transcriper")

## required Libraries

In [ ]:
import os
import sys
import time                                                                         
import datetime
import json
import platform

In [ ]:
import hardware_info

In [ ]:
py_version = sys.version
os_platform = sys.platform


py_version
os_platform

In [ ]:
import cpuinfo
import psutil
import GPUtil

def to_gb(num_bytes):
    return f"{num_bytes / (1024 ** 3):.2f} GB"

# 1. Fetch Metrics
cpu = cpuinfo.get_cpu_info()
freq = psutil.cpu_freq()
phys = psutil.cpu_count(logical=False) or 1
logi = psutil.cpu_count(logical=True) or 1
vmem = psutil.virtual_memory()
swap = psutil.swap_memory()
gpus = GPUtil.getGPUs()

# 2. Build Table Data
rows = [
    ("CPU", "Model", cpu.get("brand_raw", "Unknown")),
    ("CPU", "Architecture", f"{cpu.get('arch', 'N/A')} ({cpu.get('bits', '')}-bit)"),
    ("CPU", "Clock (Base / Live)", f"{cpu.get('hz_advertised_friendly', 'N/A')} / {freq.current:.0f} MHz" if freq else "N/A"),
    ("CPU", "Cores / Threads", f"{phys} Cores / {logi} Threads ({logi // phys} per core)"),
    ("CPU", "L2 / L3 Cache", f"{cpu.get('l2_cache_size', 'N/A')} / {cpu.get('l3_cache_size', 'N/A')}"),
    ("RAM", "Physical (Used/Total)", f"{to_gb(vmem.used)} / {to_gb(vmem.total)} ({vmem.percent}%)"),
    ("RAM", "Available / Free", f"{to_gb(vmem.available)} / {to_gb(vmem.free)}"),
    ("RAM", "Swap (Used/Total)", f"{to_gb(swap.used)} / {to_gb(swap.total)} ({swap.percent}%)"),
]

if gpus:
    for idx, gpu in enumerate(gpus, 1):
        rows.extend([
            (f"GPU {idx}", "Model / Driver", f"{gpu.name} (v{gpu.driver})"),
            (f"GPU {idx}", "VRAM (Used/Total)", f"{gpu.memoryUsed:.0f} MB / {gpu.memoryTotal:.0f} MB ({gpu.memoryUtil * 100:.1f}%)"),
            (f"GPU {idx}", "Load / Temp", f"{gpu.load * 100:.1f}% / {gpu.temperature} °C"),
        ])
else:
    rows.append(("GPU", "Status", "No NVIDIA GPU detected via GPUtil"))

# 3. Print Compact Table
print(os_platform)
print(f"{'Component':<10} | {'Metric':<22} | {'Value'}")
print("-" * 65)
for comp, metric, val in rows:
    print(f"{comp:<10} | {metric:<22} | {val}")

In [ ]:
os_platform

In [ ]:
import subprocess

try:
    output = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"])
    print(output.decode("utf-8").strip())
except (FileNotFoundError, subprocess.CalledProcessError):
    print("NVIDIA driver/GPU not found or nvidia-smi unavailable.")

In [ ]:
import torch

print("=== GPU Information ===")
print(f"CUDA Available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
    print(f"VRAM:            {vram_gb} GB")
else:
    print("Running on CPU only.")